In [91]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from matplotlib.patches import Wedge, Circle
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.colors import ListedColormap

def log_abs_values(values, eps=1e-12):
    """
    Zwraca log10(abs(values) + eps).

    Przydatne do rysowania amplitud spinów w skali logarytmicznej.
    eps zabezpiecza przed log10(0).
    """
    return np.log10(np.abs(values) + eps)

def as_bool(x):
    if isinstance(x, str):
        return x.lower() == "true"
    return bool(x)


def yperiodic_string(x):
    return "true" if as_bool(x) else "false"


def results_geometry_folder(plot_type):
    """
    Zamienia nazwę geometrii używaną w funkcjach rysujących
    na nazwę katalogu w results/.

    Przykłady:
        plot_type="pila"      -> results/saw/
        plot_type="truesnake" -> results/truesnake/
    """
    key = str(plot_type).lower()

    if key in ["pila", "piła", "bottom", "bottomup", "bottom-up", "saw"]:
        return "saw"

    if key in ["truesnake", "true_snake", "true-snake"]:
        return "truesnake"

    if key in ["snake", "semisnake", "semi-snake"]:
        return "snake"

    if key in ["snake_rows", "rows", "horizontal", "row-snake"]:
        return "snake_rows"

    raise ValueError(
        "Nieznany plot_type. Dostępne ścieżki: "
        "'saw'/'pila', 'truesnake', 'snake', 'snake_rows'."
    )


def load_component(path):
    with open(path, "r", encoding="utf-8") as f:
        header = f.readline().strip()

    mg_sites = [int(x) for x in header.replace("#", "").split() if x != "i"]

    data = np.loadtxt(path)
    sites = data[:, 0].astype(int)
    vals = data[:, 1:]

    return sites, vals, mg_sites

def shorten_segment(p1, p2, frac=0.22):
    p1 = np.array(p1, float)
    p2 = np.array(p2, float)
    d = p2 - p1
    return p1 + frac * d, p2 - frac * d


def max_scatter_s_no_overlap(ax, pos_xy, safety=0.85):
    disp = ax.transData.transform(pos_xy)

    dmin2 = np.inf
    for i in range(len(disp)):
        di = disp[i] - disp
        dist2 = di[:, 0]**2 + di[:, 1]**2
        dist2[i] = np.inf
        dmin2 = min(dmin2, dist2.min())

    dmin_px = np.sqrt(dmin2)
    diameter_px = dmin_px * safety
    radius_px = diameter_px / 2.0

    dpi = ax.figure.dpi
    radius_pt = radius_px * 72.0 / dpi

    return np.pi * radius_pt**2


def site_radius_from_geometry(pos, scale=0.33):
    dmin = np.inf

    for i in range(len(pos)):
        d = pos[i] - pos
        dist = np.sqrt(d[:, 0]**2 + d[:, 1]**2)
        dist[i] = np.inf
        dmin = min(dmin, dist.min())

    return scale * dmin


# ============================================================
#  Typy numeracji / bondów
# ============================================================

def bonds_pila(Nx, Ny, yperiodic=False):
    """
    Klasyczna numeracja typu piła / bottom-up.
    """
    Nsites = 2 * Nx * Ny
    bonds = []

    for i in range(1, Nsites):
        if i % 2 == 1:
            bonds.append((i, i + 1, "xx"))
        else:
            if i <= 2 * Ny * (Nx - 1):
                bonds.append((i, i + 2 * Ny - 1, "yy"))

            if i % (2 * Ny) != 0:
                bonds.append((i, i + 1, "zz"))
            elif yperiodic:
                col_start = i - 2 * Ny + 1
                bonds.append((i, col_start, "zz"))

    return Nsites, bonds


def site_index_snake_vertical(x, y, sub, Nx, Ny):
    """
    Semi-snake po kolumnach.
    Nieparzysta kolumna: y rośnie.
    Parzysta kolumna: y maleje.
    Kolejność A/B bez odwracania.
    """
    col_offset = 2 * Ny * (x - 1)
    yeff = y if x % 2 == 1 else Ny - y + 1

    return col_offset + 2 * (yeff - 1) + sub


def site_index_true_snake_vertical(x, y, sub, Nx, Ny):
    """
    True-snake po kolumnach.
    Nieparzysta kolumna: A, B.
    Parzysta kolumna: B, A.
    """
    col_offset = 2 * Ny * (x - 1)

    if x % 2 == 1:
        yeff = y
        subeff = sub
    else:
        yeff = Ny - y + 1
        subeff = 3 - sub

    return col_offset + 2 * (yeff - 1) + subeff


def bonds_from_indexer(Nx, Ny, yperiodic=False, indexer=site_index_snake_vertical):
    Nsites = 2 * Nx * Ny
    bonds = []

    def A(x, y):
        return indexer(x, y, 1, Nx, Ny)

    def B(x, y):
        return indexer(x, y, 2, Nx, Ny)

    for x in range(1, Nx + 1):
        for y in range(1, Ny + 1):

            bonds.append((A(x, y), B(x, y), "xx"))

            if y < Ny:
                bonds.append((B(x, y), A(x, y + 1), "zz"))
            elif yperiodic:
                bonds.append((B(x, y), A(x, 1), "zz"))

            if x < Nx:
                bonds.append((B(x, y), A(x + 1, y), "yy"))

    return Nsites, bonds


def bonds_snake(Nx, Ny, yperiodic=False):
    return bonds_from_indexer(
        Nx, Ny,
        yperiodic=yperiodic,
        indexer=site_index_snake_vertical
    )


def bonds_truesnake(Nx, Ny, yperiodic=False):
    return bonds_from_indexer(
        Nx, Ny,
        yperiodic=yperiodic,
        indexer=site_index_true_snake_vertical
    )


def site_index_snake_horizontal(x, y, sub, Nx, Ny):
    """
    Snake po wierszach.
    """
    row_offset = 2 * Nx * (y - 1)
    xeff = x if y % 2 == 1 else Nx - x + 1

    return row_offset + 2 * (xeff - 1) + sub


def bonds_snake_rows(Nx, Ny, yperiodic=False):
    """
    Snake po wierszach, zgodny z blokami długości 2*Nx.
    """
    Nsites = 2 * Nx * Ny
    bonds = []

    for i in range(1, Nsites):
        if i % 2 != 0:
            bonds.append((i, i + 1, "xx"))
        else:
            if i % (2 * Nx) != 0:
                bonds.append((i, i + 1, "yy"))

            if i <= 2 * Nx * (Ny - 1):
                bonds.append((i, i + 2 * Nx - 1, "zz"))
            elif yperiodic:
                bonds.append((i, i - 2 * Nx * (Ny - 1) + 1, "zz"))

    return Nsites, bonds


def get_bonds(Nx, Ny, plot_type="truesnake", yperiodic=False):
    plot_type = plot_type.lower()

    if plot_type in ["pila", "piła", "bottom", "bottomup", "bottom-up"]:
        return bonds_pila(Nx, Ny, yperiodic=yperiodic)

    if plot_type in ["snake", "semisnake", "semi-snake"]:
        return bonds_snake(Nx, Ny, yperiodic=yperiodic)

    if plot_type in ["truesnake", "true_snake", "true-snake"]:
        return bonds_truesnake(Nx, Ny, yperiodic=yperiodic)

    if plot_type in ["snake_rows", "rows", "horizontal", "row-snake"]:
        return bonds_snake_rows(Nx, Ny, yperiodic=yperiodic)

    raise ValueError(
        "Nieznany plot_type. Dostępne: "
        "'pila', 'snake', 'truesnake', 'snake_rows'."
    )


# ============================================================
#  Pozycje
# ============================================================

def honeycomb_positions_from_bonds(Nx, Ny, bonds):
    Nsites = 2 * Nx * Ny

    a = 1.0
    vx = (np.sqrt(3) * a,  a)
    vy = (np.sqrt(3) * a, -a)
    vz = (0.0,          2 * a)

    vmap = {
        "xx": vx,
        "yy": vy,
        "zz": vz,
    }

    adj = [[] for _ in range(Nsites + 1)]

    for s1, s2, bt in bonds:
        v = vmap[bt]
        adj[s1].append((s2, v))
        adj[s2].append((s1, (-v[0], -v[1])))

    pos = np.zeros((Nsites + 1, 2), dtype=float)
    placed = np.zeros(Nsites + 1, dtype=bool)

    pos[1] = (0.0, 0.0)
    placed[1] = True
    queue = [1]

    while queue:
        s = queue.pop(0)
        xs, ys = pos[s]

        for t, v in adj[s]:
            if not placed[t]:
                pos[t] = (xs + v[0], ys + v[1])
                placed[t] = True
                queue.append(t)

    return pos[1:]


def get_lattice_geometry(Nx, Ny, plot_type="truesnake", yperiodic=False):
    _, bonds = get_bonds(Nx, Ny, plot_type=plot_type, yperiodic=yperiodic)
    pos = honeycomb_positions_from_bonds(Nx, Ny, bonds)
    return bonds, pos

def make_current_path(
    Nx, Ny,
    yperiodic,
    Jx, Jy, Jz,
    r,
    theta, phi,
    mg_site,
    component,
    plot_type="truesnake",
    secondspin=False,
    fixed_pos=None,
):
    yp = yperiodic_string(yperiodic)
    geometry = results_geometry_folder(plot_type)

    if secondspin:
        filename = f"{component}_2spins_fixed{fixed_pos}"
    else:
        filename = component

    path = (
        f"results/{geometry}/{Nx}x{Ny}_PBC={yp}_Jx={Jx}_Jy={Jy}_Jz={Jz}_r={r}"
        f"/Savrg_theta={theta}_phi={phi}_{mg_site}/{filename}.dat"
    )

    return path

def load_spin_components(
    Nx, Ny,
    yperiodic,
    Jx, Jy, Jz,
    r,
    theta, phi,
    mg_site,
    plot_type="truesnake",
    secondspin=False,
    fixed_pos=None,
):
    """
    Wczytuje Sx, Sy, Sz dla aktualnych parametrów.

    Zwraca:
        sites, vals_x, vals_y, vals_z, mg_sites
    """

    components = ["Sx", "Sy", "Sz"]
    data = {}

    sites_ref = None
    mg_sites_ref = None

    for comp in components:
        path = make_current_path(
            Nx=Nx,
            Ny=Ny,
            yperiodic=yperiodic,
            Jx=Jx,
            Jy=Jy,
            Jz=Jz,
            r=r,
            theta=theta,
            phi=phi,
            mg_site=mg_site,
            component=comp,
            plot_type=plot_type,
            secondspin=secondspin,
            fixed_pos=fixed_pos,
        )

        sites, vals, mg_sites = load_component(path)

        if sites_ref is None:
            sites_ref = sites
            mg_sites_ref = mg_sites
        else:
            if not np.array_equal(sites, sites_ref):
                raise ValueError(f"Niezgodne indeksy site w pliku {comp}.dat")

            if mg_sites != mg_sites_ref:
                raise ValueError(f"Niezgodne mg_sites w pliku {comp}.dat")

        data[comp] = vals

    return sites_ref, data["Sx"], data["Sy"], data["Sz"], mg_sites_ref

def add_all_spin_components_legend(ax, box=(-0.21, 0.90, 0.20, 0.20)):
    """
    Rysuje w lewym górnym rogu mini-legendę:
    jedno kółko podzielone na 3 części z opisami S_x, S_y, S_z
    i literą n w środku.

    box = (left, bottom, width, height) w układzie ax.transAxes
    """

    # mała dodatkowa oś osadzona w głównej osi
    lax = ax.inset_axes(box)
    lax.set_xlim(0, 1)
    lax.set_ylim(0, 1)
    lax.set_aspect("equal")
    lax.axis("off")

    cx, cy = 0.5, 0.5
    radius = 0.43

    legend_sectors = [
        (r"$S_x$", -30,  90),
        (r"$S_y$",  90, 210),
        (r"$S_z$", 210, 330),
    ]

    for label, th1, th2 in legend_sectors:
        patch = Wedge(
            center=(cx, cy),
            r=radius,
            theta1=th1,
            theta2=th2,
            facecolor="white",
            edgecolor="black",
            linewidth=1.0,
            zorder=20,
        )
        lax.add_patch(patch)

        ang = np.deg2rad(0.5 * (th1 + th2))
        tx = cx + 0.62 * radius * np.cos(ang)
        ty = cy + 0.62 * radius * np.sin(ang)

        lax.text(
            tx,
            ty,
            label,
            ha="center",
            va="center",
            fontsize=15,
            fontweight="bold",
            color="black",
            zorder=21,
        )

    # małe białe kółko w środku
    lax.add_patch(
        Circle(
            (cx, cy),
            radius=0.28 * radius,
            facecolor="white",
            edgecolor="black",
            linewidth=0.8,
            zorder=22,
        )
    )

    lax.text(
        cx,
        cy,
        "n",
        ha="center",
        va="center",
        fontsize=15,
        fontweight="bold",
        color="black",
        zorder=23,
    )

def plot_lattice_all_spin_components(
    Nx, Ny,
    bonds,
    pos,
    vals_x,
    vals_y,
    vals_z,
    mg_sites=None,
    mg_site=None,
    fixed_pos=None,
    secondspin=False,
    mark_mg=True,
    title="",
    show_labels="auto",
    cmap="bwr",
    vmin=-0.5,
    vmax=0.5,
    radius_scale=0.33,
    log_abs=False,
    eps=1e-12,
    cbar_label=None,
    title_fontsize=16,
):
    """
    Rysuje każdy site jako kółko podzielone na trzy części:
        część x -> Sx
        część y -> Sy
        część z -> Sz

    Sektory są ustawione zgodnie z kierunkami bondów:
        xx: kierunek około +30 stopni
        yy: kierunek około -30 stopni
        zz: kierunek pionowy
    """

    Nsites = 2 * Nx * Ny

    if vals_x.ndim == 2:
        if mg_site is None:
            raise ValueError("Dla danych 2D podaj mg_site.")
        if mg_sites is None:
            raise ValueError("Dla danych 2D potrzebna jest lista mg_sites.")

        k = mg_sites.index(mg_site)
        vx = vals_x[:, k]
        vy = vals_y[:, k]
        vz = vals_z[:, k]
    else:
        vx = vals_x
        vy = vals_y
        vz = vals_z
    
    if log_abs:
        vx = log_abs_values(vx, eps=eps)
        vy = log_abs_values(vy, eps=eps)
        vz = log_abs_values(vz, eps=eps)

    fig = plt.gcf()
    ax = plt.gca()

    bond_color = {
        "xx": "blue",
        "yy": "red",
        "zz": "green",
    }

    for s1, s2, bt in bonds:
        p1 = pos[s1 - 1]
        p2 = pos[s2 - 1]
        p1s, p2s = shorten_segment(p1, p2, frac=0.22)

        ax.plot(
            [p1s[0], p2s[0]],
            [p1s[1], p2s[1]],
            lw=2,
            color=bond_color[bt],
            alpha=0.8,
            zorder=1,
        )

    norm = Normalize(vmin=vmin, vmax=vmax)
    cmap_obj = plt.get_cmap(cmap)

    radius = site_radius_from_geometry(pos, scale=radius_scale)

    sector_defs = [
        ("Sx", vx, -30,  90),    # xx
        ("Sy", vy,  90, 210),    # yy
        ("Sz", vz, 210, 330),    # zz
    ]

    for i in range(Nsites):
        x0, y0 = pos[i]

        for _, values, th1, th2 in sector_defs:
            patch = Wedge(
                center=(x0, y0),
                r=radius,
                theta1=th1,
                theta2=th2,
                facecolor=cmap_obj(norm(values[i])),
                edgecolor="k",
                linewidth=0.8,
                zorder=3,
            )
            ax.add_patch(patch)

        circ = Circle(
            (x0, y0),
            radius=radius,
            facecolor="none",
            edgecolor="k",
            linewidth=1.0,
            zorder=4,
        )
        ax.add_patch(circ)

    sm = ScalarMappable(norm=norm, cmap=cmap_obj)
    sm.set_array([])
    if cbar_label is None:
        if log_abs:
            cbar_label = r"$|S_\alpha|$"
        else:
            cbar_label = r"$\langle S_\alpha^i\rangle$"
    cbar = fig.colorbar(sm, ax=ax, label=cbar_label)

    # === ROZMIARY CZCIONEK I PODMIANA ETYKIET DLA LOG_ABS ===
    FONT_LABEL_SIZE = 20  # Wielkość czcionki głównego opisu cbar_label
    FONT_TICKS_SIZE = 15  # Wielkość czcionki wartości podziałki

    # Ustawienie rozmiaru głównej etykiety
    cbar.set_label(cbar_label, size=FONT_LABEL_SIZE, labelpad=15)
    
    if log_abs:
        import numpy as np
        # Wyciągamy granice vmin i vmax z Twojego obiektu norm
        min_val, max_val = norm.vmin, norm.vmax
        
        # Generujemy całkowite punkty logarytmiczne (np. -1, -2, -3) mieszczące się w zakresie
        tick_positions = np.arange(int(np.floor(min_val)), int(np.ceil(max_val)) + 1)
        tick_positions = [t for t in tick_positions if min_val <= t <= max_val]
        
        # Wymuszamy pozycje ticków
        cbar.set_ticks(tick_positions)
        
        # Mapujemy je na zapis potęgowy 10^-x
        tick_labels = [f"$10^{{{int(t)}}}$" if t != 0 else "$1$" for t in tick_positions]
        cbar.set_ticklabels(tick_labels)

    # Ustawienie rozmiaru czcionki dla wartości na osi colorbaru (działa dla obu skal)
    cbar.ax.tick_params(labelsize=FONT_TICKS_SIZE)
    add_all_spin_components_legend(ax, box=(-0.05, 0.75, 0.3, 0.3))

    if show_labels == "auto":
        do_labels = Nsites <= 120
    else:
        do_labels = bool(show_labels)

    if do_labels:
        label_radius = 0.45 * radius

        for s in range(1, Nsites + 1):
            x0, y0 = pos[s - 1]

            ax.add_patch(
                Circle(
                    (x0, y0),
                    radius=label_radius,
                    facecolor="white",
                    edgecolor="black",
                    linewidth=0.6,
                    zorder=5,
                )
            )

            ax.text(
                x0,
                y0,
                str(s),
                ha="center",
                va="center",
                fontsize=12,
                fontweight="bold",
                color="black",
                zorder=6,
            )

    if mark_mg and mg_site is not None:
        ax.add_patch(
            Circle(
                (pos[mg_site - 1, 0], pos[mg_site - 1, 1]),
                radius=1.28 * radius,
                facecolor="none",
                edgecolor="k",
                linewidth=3,
                zorder=6,
            )
        )

    if secondspin and fixed_pos is not None:
        ax.add_patch(
            Circle(
                (pos[fixed_pos - 1, 0], pos[fixed_pos - 1, 1]),
                radius=1.15 * radius,
                facecolor="none",
                edgecolor="k",
                linewidth=3,
                zorder=7,
            )
        )

    xmin, ymin = pos.min(axis=0)
    xmax, ymax = pos.max(axis=0)

    dx = 12.0#xmax - xmin
    dy = 6.2#ymax - ymin
    xpad = max(0.10 * dx, 0.8)
    ypad = max(0.20 * dy, 0.8)

    ax.set_xlim(xmin - xpad, xmax + xpad)
    ax.set_ylim(ymin - ypad, ymax + ypad)

    ax.set_aspect("equal", adjustable="box")
    ax.axis("off")

    if mg_site is None:
        ax.set_title(title)
    else:
        extra = f" (magnetic site = {mg_site})"

        if secondspin and fixed_pos is not None:
            extra += f", fixed = {fixed_pos}"

        ax.set_title(f"{title}", fontsize=title_fontsize)



def plot_current_all_spin_components(
    plot_type="truesnake",
    show_labels="auto",
    cmap="bwr",
    vmin=-0.5,
    vmax=0.5,
    radius_scale=0.53,
    log_abs=False,
    eps=1e-12,
    cbar_label=None,
    title=None,
    title_fontsize=16,
):
    """
    Rysuje Sx, Sy, Sz jednocześnie.
    Każdy site jest kółkiem podzielonym na trzy sektory.

    Wywołanie na dole notebooka:
        plot_current_all_spin_components(plot_type="truesnake")
    """

    yp_bool = as_bool(yperiodic)

    bonds, pos = get_lattice_geometry(
        Nx,
        Ny,
        plot_type=plot_type,
        yperiodic=yp_bool,
    )

    _, vals_x, vals_y, vals_z, mg_sites = load_spin_components(
        Nx=Nx,
        Ny=Ny,
        yperiodic=yperiodic,
        Jx=Jx,
        Jy=Jy,
        Jz=Jz,
        r=r,
        theta=theta,
        phi=phi,
        mg_site=mg_site,
        plot_type=plot_type,
        secondspin=secondspin,
        fixed_pos=fixed_pos,
    )

    xmin, ymin = pos.min(axis=0)
    xmax, ymax = pos.max(axis=0)

    width = xmax - xmin
    height = ymax - ymin
    aspect = width / height

    base = 9

    if aspect > 1:
        fig_w = base
        fig_h = base / aspect
    else:
        fig_h = base
        fig_w = base * aspect

    plt.figure(figsize=(fig_w, fig_h), constrained_layout=True)

    plot_lattice_all_spin_components(
        Nx,
        Ny,
        bonds,
        pos,
        vals_x,
        vals_y,
        vals_z,
        mg_sites=mg_sites,
        mg_site=mg_site,
        fixed_pos=fixed_pos,
        secondspin=secondspin,
        title=title,
        show_labels=show_labels,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        radius_scale=radius_scale,
        log_abs=log_abs,
        eps=eps,
        cbar_label=cbar_label,
        title_fontsize=title_fontsize,
    )

    plt.show()

In [8]:
"""
    Rysuje wykres dla aktualnie ustawionych zmiennych:
    Nx, Ny, yperiodic, Jx, Jy, Jz, r, theta, phi,
    component, mg_site, secondspin, fixed_pos.

    plot_type:
        "pila"
        "snake"
        "truesnake"
        "snake_rows"
    """

Nx = 5
Ny = 4

secondspin = False
fixed_pos = 2

component = "Sx"
mg_site = 20

yperiodic = "false"

Jx = -1.0
Jy = -1.0
Jz = -1.0
r = 0.0

theta = 0.0
phi = 0.0

In [ ]:
plot_current_all_spin_components(
    plot_type="pila",
    log_abs=True,
    vmin=-8,
    vmax=0,
    radius_scale=0.5,
    cmap="OrRd",
    title="b)",
    title_fontsize=30,
)

ValueError: Nieznany plot_type. Dostępne: 'pila', 'snake', 'truesnake', 'snake_rows'.